In [ ]:
import pandas as pd
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *

source_path = '/Users/harryclark/Downloads/COHORT12/'

mouse_days = {
    20: [14,15,16,17,18,19,20,21,22,23,24,25,26],
    21: [15,16,17,18,19,20,21,22,23,24,25,26],
    22: [33,34,35,36,37,38,39,40,41],
    25: [16,17,18,19,20,21,22,23,24,25],
    26: [11,12,13,14,15,16,17,18,19],
    27: [16,17,18,19,20,21,22,23,24,26],
    28: [16,17,18,19,20,21,22,23,25],
    29: [16,17,18,19,20,21,22,23,25],
}

for use_brain_regions, ubr_str in zip([True, False],['', '_no_regions']):
    concatenated_results = []
    for mouse, days in mouse_days.items():
        for day in days:
            print(f"Processing M{mouse} D{day:02}...")

            if use_brain_regions:
                gcs, ngs, all_cells = classify_cells_both_sessions(
                    mouse, day, percentile_threshold=95, source_path=None, verbose=False,
                )
            else:
                gcs, ngs, all_cells = classify_cells_both_sessions(
                    mouse, day, percentile_threshold=95, source_path=None, verbose=False,
                    disqualifying_regions=other_areas,
                )

            # ── BUG FIX: use cluster_id values not row index for exclusion ────────
            # cluster_id is the DataFrame INDEX in classify_cells_both_sessions output,
            # not a column. all_cells may have a numeric reset index while gcs/ngs
            # retain cluster_id as index — so all_cells.index.isin(gcs.index) was
            # always False, and every cell ended up in 'others' as well → duplicates.
            def _get_ids(df):
                """Get cluster_ids whether stored as index or column."""
                if 'cluster_id' in df.columns:
                    return set(df['cluster_id'].values)
                return set(df.index.values)

            gc_ids  = _get_ids(gcs)
            ngs_ids = _get_ids(ngs)

            if 'cluster_id' in all_cells.columns:
                _excl_mask = (~all_cells['cluster_id'].isin(gc_ids) &
                              ~all_cells['cluster_id'].isin(ngs_ids))
            else:
                _excl_mask = (~all_cells.index.isin(gc_ids) &
                              ~all_cells.index.isin(ngs_ids))
            others = all_cells[_excl_mask].copy()

            gcs    = gcs.copy()
            ngs    = ngs.copy()

            gcs['cell_type']    = 'GC'
            ngs['cell_type']    = 'NG'
            others['cell_type'] = 'Other'

            # --- HDBSCAN grid modules ---
            gcs['grid_module']    = pd.NA
            ngs['grid_module']    = pd.NA
            others['grid_module'] = pd.NA

            try:
                g_m_ids, g_m_cluster_ids, _ = HDBSCAN_grid_modules(
                    gcs, all_cells, mouse, day,
                    min_cluster_size=3, cluster_selection_epsilon=3,
                    curate_with_vr=False, curate_with_brain_region=True,
                    figpath='', verbose=False,
                )
                for module_idx, cluster_ids in enumerate(g_m_cluster_ids):
                    mask = gcs['cluster_id'].isin(cluster_ids)
                    gcs.loc[mask, 'grid_module'] = module_idx
            except Exception as e:
                print(f"  WARNING: HDBSCAN failed for M{mouse} D{day}: {e}")

            combined = pd.concat([gcs, ngs, others], ignore_index=True)

            # Per-session assertion: exactly 1 row per cluster_id
            n_dupes = combined.duplicated('cluster_id').sum()
            if n_dupes > 0:
                print(f"  ERROR M{mouse} D{day}: {n_dupes} duplicate cluster_ids after dedup!")
            else:
                print(f"  ✓ M{mouse} D{day}: {len(combined)} unique cells  "
                      f"(GC={len(combined[combined.cell_type=='GC'])}  "
                      f"NG={len(combined[combined.cell_type=='NG'])}  "
                      f"Other={len(combined[combined.cell_type=='Other'])})")

            # --- VR classification ---
            combined['ramp_cell']                   = False
            combined['speed_modulated']             = False
            combined['vr_outbound_sign']            = pd.NA
            combined['vr_homebound_sign']           = pd.NA
            combined['vr_spatial_information']      = pd.NA
            combined['vr_spatial_information_sig']  = pd.NA

            combined = reconstruct_shank_id(combined, mouse, colname='probe_x')

            try:
                ramp_cells, ramp_and_speed_cells, non_spatial_cells = cell_classification_vr(
                    mouse, day, percentile_threshold=99, source_path=None,
                )

                if len(ramp_cells) > 0:
                    ramp_idx         = ramp_cells.set_index('cluster_id')
                    speed_mod_lookup = ramp_idx['speed_modulated']
                    out_sign_lookup  = ramp_idx['ramp_class'].str[0]
                    home_sign_lookup = ramp_idx['ramp_class'].str[1]

                    mask = combined['cluster_id'].isin(ramp_idx.index)
                    combined.loc[mask, 'ramp_cell']         = True
                    combined.loc[mask, 'speed_modulated']   = combined.loc[mask, 'cluster_id'].map(speed_mod_lookup)
                    combined.loc[mask, 'vr_outbound_sign']  = combined.loc[mask, 'cluster_id'].map(out_sign_lookup)
                    combined.loc[mask, 'vr_homebound_sign'] = combined.loc[mask, 'cluster_id'].map(home_sign_lookup)

            except Exception as e:
                print(f"  WARNING: VR classification failed for M{mouse} D{day}: {e}")

            # --- VR spatial information ---
            try:
                spatial_path = f'{source_path}M{mouse}/D{day:02}/VR/tuning_scores/spatial_information.parquet'
                spatial_table = pd.read_parquet(spatial_path).set_index('cluster_id')
                si_lookup  = spatial_table['spatial_information']
                sig_lookup = spatial_table['sig']

                mask = combined['cluster_id'].isin(spatial_table.index)
                combined.loc[mask, 'vr_spatial_information']     = combined.loc[mask, 'cluster_id'].map(si_lookup)
                combined.loc[mask, 'vr_spatial_information_sig'] = combined.loc[mask, 'cluster_id'].map(sig_lookup)
            except Exception as e:
                print(f"  WARNING: Spatial information failed for M{mouse} D{day}: {e}")

            # --- classified_by: set 'neither' for Other cells ─────────────────
            # GC and NGS already have classified_by from classify_cells_both_sessions
            if 'classified_by' not in combined.columns:
                combined['classified_by'] = pd.NA
            combined.loc[combined['cell_type'] == 'Other', 'classified_by'] = 'neither'

            # --- Head direction cell classification ─────────────────────────────
            # HD information has no expected optimal spatial shift, so we
            # always use travel=0 (no lag) for classification, comparing
            # hd_information(travel=0) > null_hd_information(travel=0).
            hd_passes = {}  # cluster_id → {'OF1': bool, 'OF2': bool}
            for session in ['OF1', 'OF2']:
                hd_path = (f'{source_path}M{mouse}/D{day:02}/{session}/'
                           f'tuning_scores/shifted_hd_information.parquet')
                try:
                    hd_df    = pd.read_parquet(hd_path)
                    hd_score = hd_df.set_index(['cluster_id', 'travel'])['hd_information']
                    hd_null  = hd_df.set_index(['cluster_id', 'travel'])['null_hd_information']
                    for cid in combined['cluster_id'].astype(int).unique():
                        try:
                            score    = hd_score.loc[(cid, 0)]
                            null_val = hd_null.loc[(cid, 0)]
                            score_f  = float(score)
                            # null may be a scalar OR an array of shuffle scores
                            if hasattr(null_val, '__len__'):
                                null_arr = np.array(null_val, dtype=float)
                                null_arr = null_arr[~np.isnan(null_arr)]
                                threshold = float(np.nanpercentile(null_arr, 95)) if len(null_arr) > 0 else np.nan
                            else:
                                threshold = float(null_val) if pd.notna(null_val) else np.nan
                            passes = pd.notna(score_f) and pd.notna(threshold) and score_f > threshold
                        except KeyError:
                            passes = False
                        if cid not in hd_passes:
                            hd_passes[cid] = {}
                        hd_passes[cid][session] = passes
                except FileNotFoundError:
                    pass
                except Exception as e:
                    print(f"  WARNING: HD info failed ({session}) for M{mouse} D{day}: {e}")

            def _hd_classified_by(cid):
                entry = hd_passes.get(int(cid), {})
                sig1  = entry.get('OF1', False)
                sig2  = entry.get('OF2', False)
                if sig1 and sig2: return 'both'
                if sig1:          return 'OF1'
                if sig2:          return 'OF2'
                return 'neither'

            combined['head_direction_cell']          = combined['cluster_id'].apply(
                lambda cid: _hd_classified_by(cid) != 'neither')
            combined['head_direction_classified_by'] = combined['cluster_id'].apply(
                _hd_classified_by)

            concatenated_results.append(combined)

    results = pd.concat(concatenated_results, ignore_index=True)

    # ── Final confirmation: exactly 1 row per (mouse, day, cluster_id) ────────
    total_dupes = results.duplicated(['mouse', 'day', 'cluster_id']).sum()
    if total_dupes > 0:
        bad = results[results.duplicated(['mouse', 'day', 'cluster_id'], keep=False)]
        print(f"\n  ERROR: {total_dupes} duplicate (mouse, day, cluster_id) rows found!")
        print(bad[['mouse','day','cluster_id','cell_type']].head(10))
    else:
        print(f"\n✓ Confirmed: all {len(results):,} rows have unique (mouse, day, cluster_id) — 1 row per cell")

    results.to_csv(f'/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications{ubr_str}.csv',
                   index=False)

    print(f"Saved: cell_classifications{ubr_str}.csv")
    print(
        f'Total cells: {len(results):,}  |  '
        f'GC: {(results.cell_type=="GC").sum()}  |  '
        f'NG: {(results.cell_type=="NG").sum()}  |  '
        f'Other: {(results.cell_type=="Other").sum()}'
    )
    print(f'GCs with a grid module assigned: {results["grid_module"].notna().sum()}')
    conjunctive = results[(results['cell_type'] == 'GC') & (results['head_direction_cell'] == True)]
    print(f'Conjunctive grid cells (GC + HD): {len(conjunctive)}  '
          f'({100*len(conjunctive)/max(1,(results.cell_type=="GC").sum()):.1f}% of GCs)')
    print(f'  HD classified by: {conjunctive["head_direction_classified_by"].value_counts().to_dict()}')
    print(f'\nramp_cell counts:\n{results["ramp_cell"].value_counts()}')
    print(f'\nvr_spatial_information non-null: {results["vr_spatial_information"].notna().sum()}')


---
## Methods: Identification of Spatially and Directionally Tuned Cells

### Open-field recording sessions

Spatial and directional tuning was assessed from recordings made in an open-field arena across two sessions (OF1 and OF2). For all analyses, only periods when the animal was moving were included. Two-dimensional firing rate maps were constructed by binning spikes according to the animal's head position and normalising by occupancy time, followed by Gaussian smoothing.

### Two-dimensional firing rate maps

Two-dimensional firing rate maps were constructed by dividing the arena into a 40 × 40 grid of spatial bins (bin size ~2.5 cm, covering a 0–100 cm × 0–100 cm arena). For each bin, the total spike count was divided by the cumulative occupancy time to yield a mean firing rate. Only time points during which the animal was moving were included in both the spike count and the occupancy, preventing distortion of the rate map by long stationary periods. Rate maps were smoothed using a Gaussian kernel with a standard deviation of 1.5 bins (~3.75 cm) applied independently to the spike count and occupancy maps before division, handling unvisited bins (zero occupancy) as not-a-number (NaN).

Head direction tuning curves were constructed equivalently, binning spikes and occupancy into 20 directional bins spanning −π to π radians, without smoothing.

### Spatial travel lag correction

To account for a systematic offset between the time of spiking and the animal's physical position that best predicts spatial firing, each cell's firing was evaluated across a range of spatial travel lags (−50 to +49 cm, in 1 cm steps). At each lag, the animal's position time series was shifted before constructing the rate map. The lag that maximised the grid score was defined as the optimal travel lag for that cell and used for subsequent grid cell and spatial cell classification.

### Null distributions

For each cell, a null distribution of tuning scores was generated by circularly shifting the spike train in time by a random amount (minimum shift 20 s) and recomputing the tuning score. This was repeated 200 times. The 95th percentile of the resulting null distribution — evaluated at travel lag = 0 — was used as the significance threshold.

### Grid score

Spatial periodicity was quantified using the grid score, computed from the two-dimensional spatial autocorrelogram of the rate map. The autocorrelogram was calculated as the Pearson correlation between the rate map and spatially offset copies of itself across all lag combinations, requiring a minimum of 20 co-sampled bins at each offset. The six peaks closest to the central peak were identified, and an ellipse-to-circle transform was applied to correct for any elongation of the grid pattern. A ring-shaped region was extracted between 50% and 125% of the mean distance to these six peaks. The grid score was defined as the difference between the minimum Pearson correlation when the ring was rotated by 60° and 120° (rotational symmetry consistent with a hexagonal grid) and the maximum correlation when rotated by 30°, 90°, and 150° (asymmetric rotations):

$$\text{grid score} = \min(r_{60°},\, r_{120°}) - \max(r_{30°},\, r_{90°},\, r_{150°})$$

### Spatial information

Spatial selectivity was quantified as the Shannon mutual information between the animal's position and the cell's firing rate (bits per spike), computed from the two-dimensional rate map using the full 2D position signal.

### Head direction tuning

Head direction selectivity was quantified as the mean resultant length (MRL) of the head direction tuning curve, a circular statistic that measures the concentration of firing around a preferred direction. For a tuning curve with firing rate $r(\theta_i)$ at head direction $\theta_i$:

$$\text{MRL} = \sqrt{\left(\frac{\sum_i r(\theta_i)\cos\theta_i}{\sum_i r(\theta_i)}\right)^2 + \left(\frac{\sum_i r(\theta_i)\sin\theta_i}{\sum_i r(\theta_i)}\right)^2}$$

The MRL ranges from 0 (uniform tuning) to 1 (perfectly concentrated at one direction). Unlike spatial tuning, head direction tuning is not expected to benefit from a spatial travel lag; MRL was therefore computed at travel lag = 0 only.

### Cell classification

**Grid cells (GC):** a cell was classified as a grid cell if, at its optimal travel lag, both the grid score and the spatial information exceeded the 95th percentile of their respective null distributions (evaluated at lag = 0), in at least one open-field session.

**Non-grid spatial cells (NGS):** a cell was classified as a non-grid spatial cell if its spatial information at the optimal travel lag exceeded the 95th percentile of its null distribution, but it did not meet the grid cell criteria.

**Non-spatial cells:** cells that met neither criterion were classified as non-spatial.

For both grid and non-grid spatial cells, the `classified_by` field records whether the cell met criteria in both sessions, in OF1 only, or in OF2 only.

**Head direction cells:** a cell was classified as a head direction cell if its MRL at travel lag = 0 exceeded the 95th percentile of its circular-shuffle null distribution, in at least one session. Head direction classification is independent of spatial classification; cells that satisfy both grid cell and head direction criteria are referred to as conjunctive grid cells.

### Grid module assignment

Grid cells were grouped into co-modular ensembles using HDBSCAN density-based clustering applied to each cell's grid orientation and field spacing. Cells assigned to the same cluster are considered to share a common spatial phase offset and are putatively driven by a single continuous attractor network module.
